# Collect and label AI responses
Run top to bottom. **Kernel:** the project Python environment with pandas installed. First time: copy `track3/.env.example` to `track3/.env` and fill in the `RPSD_*` Google IDs.

| Step | Costs money? |
|---|---|
| 1 Settings · 2 Keys · 3 Status | No |
| 4 Collect answers from the APIs | **Yes**, only if `ALLOW_SPENDING = True` |
| 5 Quick look · 6 Load fact sheet + sheet | No |
| 7 Auto-label + fact-check | **Yes** for new answers, only if `ALLOW_SPENDING = True` (saved labels are free) |
| 8 Export + compare · 9 Inspect | No |

With `ALLOW_SPENDING = False` (the default), every step still runs. Paid steps only report what they *would* send. Anything already collected or labeled is never paid for again.

- **Study rules** (questions, labeling instructions, clinic names, accuracy wording): `track3/rpsd_rules.py`
- **Code:** `track3/rpsd_tools.py`
- **Results:** `RPSD-Capstone/data/track3/` (gitignored, never committed)
- Keys stay in this session only. Restart the kernel to clear them. API results may differ from the consumer apps. No user location is sent.

## 1 · Settings

In [ ]:
import sys, importlib
from pathlib import Path
from IPython.display import display
REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'track3' / 'rpsd_tools.py').is_file()), None)
if REPO is None:
    raise FileNotFoundError('Start this notebook inside the RPSD-Capstone repository.')
HERE = REPO / 'track3'
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))
import rpsd_rules, rpsd_tools
_kept = (rpsd_tools.KEYS, rpsd_tools.MODEL_LISTS, rpsd_tools.ACCESS)      # keep keys when re-running this cell
importlib.reload(rpsd_rules); T = importlib.reload(rpsd_tools)
T.KEYS.update(_kept[0]); T.MODEL_LISTS.update(_kept[1]); T.ACCESS.update(_kept[2])
ENV = T.load_env()            # track3/.env (gitignored): private Google IDs, see track3/.env.example

T.configure(
    ALLOW_SPENDING = False,          # <- money switch. False = free mode: nothing paid is sent.

    BATCH_ID = 'api-pilot-01',       # change only for a NEW batch; re-running resumes this one
    TRIALS = 3,

    # Same tier per provider: the everyday default a free user gets in that company's app.
    MODELS = {
        'OpenAI': 'chat-latest',         # OpenAI's alias for the model currently used in ChatGPT
        'Claude': 'claude-sonnet-5-5',   # Claude.ai's default tier (Sonnet)
        'Gemini': 'gemini-3.8-flash',    # Gemini app's default tier (Flash)
    },

    # Fact-check judge: one fixed model for every answer, so labels are comparable.
    JUDGE_PROVIDER = 'OpenAI',
    JUDGE_MODEL = 'gpt-4.1',

    SHEET_ID = ENV.get('RPSD_TRACKER_SHEET_ID', ''),                     # Pilot Testing 1 (from .env)
    SHEET_TABS = T.sheet_tabs_from_env(ENV.get('RPSD_TRACKER_TAB_GIDS')), # tab name -> gid (from .env)
    FACT_DOC_ID = ENV.get('RPSD_FACT_SHEET_DOC_ID', ''),                 # RPSD Fact Sheet (from .env)
    LABEL_API_RESULTS = True,
    MAX_ANSWERS = None,              # None = all answers found in step 6; a number caps paid labeling
)
print('Kernel:', sys.executable)
if not (T.SHEET_ID and T.FACT_DOC_ID and T.SHEET_TABS):
    print('  Missing Google IDs: copy track3/.env.example to track3/.env and fill in the RPSD_* lines.')
print(f'Free mode: {not T.ALLOW_SPENDING} | rules {rpsd_rules.RULES_VERSION} | results in {T.DATA}')

## 2 · Keys
A hidden prompt for each provider. Press Enter to skip one, or to keep a key you entered earlier. Use keys from the clinic's **Track 3** project. Checking a key only lists models, which is free. Use `show_models=True` to print the lists.

In [ ]:
T.enter_keys(show_models=False)

## 3 · Status

In [ ]:
T.status()

## 4 · Collect answers
6 questions (`rpsd_rules.QUESTIONS`) × `TRIALS` per ready provider, with web search on. Each call is a fresh, single-question conversation. Saved question/trial slots are skipped. In free mode this only counts what would be sent.

In [ ]:
T.collect()

## 5 · Quick look

In [ ]:
display(T.results_table())
T.show_answer('OpenAI', 'P01', 1)      # change to read another answer

## 6 · Load the fact sheet and the sheet's answers
Both are read live from Google, so the files must stay shared as "anyone with the link". The fact sheet is numbered F01, F02, … and saved as a versioned snapshot. Hand labels already in the sheet are kept for the comparison in step 8.

In [ ]:
display(T.load_fact_sheet())
print(T.gather_items())

## 7 · Auto-label and fact-check
Labels up to `MAX_ANSWERS` answers. Saved labels are reused for free. New ones need two judge calls each and run only with `ALLOW_SPENDING = True`. Every auto-label is a draft for human review.

In [ ]:
T.run_labels()

## 8 · Export, and compare visibility labels with hand labels
Saves tracker CSVs to `data/`:
- `autolabels-…csv`: the auto-labels.
- `handlabels-normalized-<tab>-…csv`: your existing labels converted to the standard values, with each change listed in Notes.

To paste them into the sheet: first back up the tab and add column **V "Numbered-list position"**. Then paste L–S and V over the matching rows.

In [ ]:
cmp = T.export_and_compare()
display(cmp)

## 9 · Inspect one labeled answer

In [ ]:
T.inspect('S001')      # any Run ID from step 7